# Task 3: Feature Engineering

## Credit Risk Probability Model Using Alternative Data

### Objective

The goal of this notebook is to transform raw transaction-level data into meaningful customer-level features that can be used for credit risk modeling.

The notebook explores feature creation strategies before implementing the final production pipeline in `src/data_processing.py`.

## 1. Import Libraries

Import all libraries required for data manipulation, feature engineering, and preprocessing.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

## 2. Load Dataset

Load the raw transaction dataset that will be used throughout the feature engineering process.

In [2]:
df = pd.read_csv("../data/raw/data.csv")

df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


### Dataset Shape

In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 95662
Columns: 16


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95662 entries, 0 to 95661
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TransactionId         95662 non-null  str    
 1   BatchId               95662 non-null  str    
 2   AccountId             95662 non-null  str    
 3   SubscriptionId        95662 non-null  str    
 4   CustomerId            95662 non-null  str    
 5   CurrencyCode          95662 non-null  str    
 6   CountryCode           95662 non-null  int64  
 7   ProviderId            95662 non-null  str    
 8   ProductId             95662 non-null  str    
 9   ProductCategory       95662 non-null  str    
 10  ChannelId             95662 non-null  str    
 11  Amount                95662 non-null  float64
 12  Value                 95662 non-null  int64  
 13  TransactionStartTime  95662 non-null  str    
 14  PricingStrategy       95662 non-null  int64  
 15  FraudResult           95662 no

### 3. Customer-Level Aggregation

Credit risk is assessed at the customer level rather than the transaction level.

Since the dataset contains transaction-level records, I need to aggregate transactions belonging to the same customer to create meaningful behavioral features.

The following features will be created:

- Total Transaction Amount
- Average Transaction Amount
- Transaction Count
- Standard Deviation of Transaction Amounts

In [5]:
customer_features = df.groupby("CustomerId").agg(
    total_transaction_amount=("Amount", "sum"),
    average_transaction_amount=("Amount", "mean"),
    transaction_count=("Amount", "count"),
    std_transaction_amount=("Amount", "std")
).reset_index()

customer_features.head()

,CustomerId,total_transaction_amount,average_transaction_amount,transaction_count,std_transaction_amount
0,CustomerId_1,-10000.0,-10000.000000,1,NaN
1,CustomerId_10,-10000.0,-10000.000000,1,NaN
2,CustomerId_1001,20000.0,4000.000000,5,6558.963333
3,CustomerId_1002,4225.0,384.090909,11,560.498966
4,CustomerId_1003,20000.0,3333.333333,6,6030.478146


## Observation

Customer-level aggregation successfully transformed the transaction-level dataset into a customer-level dataset.These features summarize customer spending behavior and may provide useful signals for future credit risk prediction.

In [6]:
customer_features.shape
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 3742 entries, 0 to 3741
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   CustomerId                  3742 non-null   str    
 1   total_transaction_amount    3742 non-null   float64
 2   average_transaction_amount  3742 non-null   float64
 3   transaction_count           3742 non-null   int64  
 4   std_transaction_amount      3030 non-null   float64
dtypes: float64(3), int64(1), str(1)
memory usage: 146.3 KB


### Observation

Customer-level aggregation reduced the dataset from 95,662 transaction records to 3,742 unique customers.

The newly created features summarize customer spending behavior and transaction activity.

The `std_transaction_amount` feature contains missing values for customers with only one transaction, since standard deviation cannot be calculated from a single observation.

# 4. Date Feature Extraction

Transaction timestamps may contain useful behavioral information.

To capture temporal patterns, the following features will be extracted from `TransactionStartTime`:

- Transaction Hour
- Transaction Day
- Transaction Month
- Transaction Year

In [7]:
df["TransactionStartTime"] = pd.to_datetime(
    df["TransactionStartTime"]
)

df["TransactionStartTime"].head()

0   2018-11-15 02:18:49+00:00
1   2018-11-15 02:19:08+00:00
2   2018-11-15 02:44:21+00:00
3   2018-11-15 03:32:55+00:00
4   2018-11-15 03:34:21+00:00
Name: TransactionStartTime, dtype: datetime64[us, UTC]

In [8]:
df["transaction_hour"] = df["TransactionStartTime"].dt.hour
df["transaction_day"] = df["TransactionStartTime"].dt.day
df["transaction_month"] = df["TransactionStartTime"].dt.month
df["transaction_year"] = df["TransactionStartTime"].dt.year

df[
    [
        "TransactionStartTime",
        "transaction_hour",
        "transaction_day",
        "transaction_month",
        "transaction_year"
    ]
].head()

,TransactionStartTime,transaction_hour,transaction_day,transaction_month,transaction_year
0,2018-11-15 02:18:49+00:00,2,15,11,2018
1,2018-11-15 02:19:08+00:00,2,15,11,2018
2,2018-11-15 02:44:21+00:00,2,15,11,2018
3,2018-11-15 03:32:55+00:00,3,15,11,2018
4,2018-11-15 03:34:21+00:00,3,15,11,2018


### Observation

These features may help capture customer behavioral patterns such as preferred transaction hours, seasonal activity, and long-term transaction trends.

## 5. Identify Categorical Features

In [10]:
df.select_dtypes(include=["object", "string"]).columns.tolist()

['TransactionId',
 'BatchId',
 'AccountId',
 'SubscriptionId',
 'CustomerId',
 'CurrencyCode',
 'ProviderId',
 'ProductId',
 'ProductCategory',
 'ChannelId']

## 6. Cardinality Analysis

Before encoding categorical variables, it is important to understand how many unique categories exist in each feature.

Features with very large numbers of unique values may require special handling.

In [11]:
categorical_cols = [
    "CurrencyCode",
    "ProviderId",
    "ProductId",
    "ProductCategory",
    "ChannelId"
]

for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

CurrencyCode: 1 unique values
ProviderId: 6 unique values
ProductId: 23 unique values
ProductCategory: 9 unique values
ChannelId: 4 unique values


## 7. Categorical Feature Selection

Based on the cardinality analysis:

Features selected for encoding:

- ProviderId
- ProductId
- ProductCategory
- ChannelId

Feature excluded:

- CurrencyCode (only one unique value)

Identifier columns such as TransactionId, BatchId, AccountId, SubscriptionId, and CustomerId will not be used as predictive features.

In [12]:
from sklearn.preprocessing import OneHotEncoder

In [13]:
categorical_features = [
    "ProviderId",
    "ProductId",
    "ProductCategory",
    "ChannelId"
]

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded_data = encoder.fit_transform(
    df[categorical_features]
)

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_features)
)

encoded_df.head()

,ProviderId_ProviderId_1,ProviderId_ProviderId_2,ProviderId_ProviderId_3,ProviderId_ProviderId_4,ProviderId_ProviderId_5,ProviderId_ProviderId_6,ProductId_ProductId_1,ProductId_ProductId_10,ProductId_ProductId_11,ProductId_ProductId_12,ProductId_ProductId_13,ProductId_ProductId_14,ProductId_ProductId_15,ProductId_ProductId_16,ProductId_ProductId_19,ProductId_ProductId_2,ProductId_ProductId_20,ProductId_ProductId_21,ProductId_ProductId_22,ProductId_ProductId_23,ProductId_ProductId_24,ProductId_ProductId_27,ProductId_ProductId_3,ProductId_ProductId_4,ProductId_ProductId_5,ProductId_ProductId_6,ProductId_ProductId_7,ProductId_ProductId_8,ProductId_ProductId_9,ProductCategory_airtime,ProductCategory_data_bundles,ProductCategory_financial_services,ProductCategory_movies,ProductCategory_other,ProductCategory_ticket,ProductCategory_transport,ProductCategory_tv,ProductCategory_utility_bill,ChannelId_ChannelId_1,ChannelId_ChannelId_2,ChannelId_ChannelId_3,ChannelId_ChannelId_5
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [14]:
customer_features.isnull().sum()

CustomerId                      0
total_transaction_amount        0
average_transaction_amount      0
transaction_count               0
std_transaction_amount        712
dtype: int64

# 8. Missing Value Handling

The aggregated customer dataset contains missing values in the `std_transaction_amount` feature.

These missing values occur because some customers performed only a single transaction, making it impossible to calculate a standard deviation.

The missing values will be replaced with 0, representing no observed transaction variability.

In [15]:
customer_features["std_transaction_amount"] = (
    customer_features["std_transaction_amount"]
    .fillna(0)
)

customer_features.isnull().sum()

CustomerId                    0
total_transaction_amount      0
average_transaction_amount    0
transaction_count             0
std_transaction_amount        0
dtype: int64

### Observation
A value of 0 was assigned to customers with a single transaction, indicating no measurable transaction variability.

# 9. Numerical Feature Scaling

Before applying scaling, I inspect the ranges of numerical features to determine whether they operate on different scales.

In [16]:
customer_features.describe()

,total_transaction_amount,average_transaction_amount,transaction_count,std_transaction_amount
count,3.742000e+03,3.742000e+03,3742.000000,3.742000e+03
mean,1.717377e+05,1.571562e+04,25.564404,1.360517e+04
std,2.717305e+06,1.676991e+05,96.929602,9.689344e+04
min,-1.049000e+08,-4.250000e+05,1.000000,0.000000e+00
25%,4.077438e+03,1.000000e+03,2.000000,5.011411e+02
50%,2.000000e+04,2.583846e+03,7.000000,3.184898e+03
75%,7.996775e+04,4.877614e+03,20.000000,6.745369e+03
max,8.345124e+07,8.601821e+06,4091.000000,3.309916e+06


# 10. Standardization of Numerical Features

To ensure that no feature dominates the model solely because of its magnitude, standardization is applied using StandardScaler.
Standardization transforms each feature to have:
- Mean = 0
- Standard Deviation = 1

In [17]:
from sklearn.preprocessing import StandardScaler

In [18]:
numerical_features = [
    "total_transaction_amount",
    "average_transaction_amount",
    "transaction_count",
    "std_transaction_amount"
]

In [19]:
scaler = StandardScaler()

scaled_data = scaler.fit_transform(
    customer_features[numerical_features]
)

scaled_df = pd.DataFrame(
    scaled_data,
    columns=numerical_features
)

scaled_df.head()

,total_transaction_amount,average_transaction_amount,transaction_count,std_transaction_amount
0,-0.066891,-0.153364,-0.253459,-0.140432
1,-0.066891,-0.153364,-0.253459,-0.140432
2,-0.055849,-0.069870,-0.212186,-0.072731
3,-0.061655,-0.091435,-0.150278,-0.134647
4,-0.055849,-0.073846,-0.201868,-0.078186


# 11. Weight of Evidence (WoE) and Information Value (IV)

Weight of Evidence (WoE) and Information Value (IV) are commonly used techniques in credit risk modeling.

## What is WoE?

Weight of Evidence transforms predictor variables into values that represent their relationship with a target outcome. It helps convert raw feature values into risk-oriented signals that are easier to interpret and analyze.

## What is IV?

Information Value measures the predictive strength of a variable.

General interpretation:

| IV Range | Predictive Power |
|-----------|------------------|
| < 0.02 | Not Predictive |
| 0.02 - 0.1 | Weak Predictor |
| 0.1 - 0.3 | Medium Predictor |
| 0.3 - 0.5 | Strong Predictor |
| > 0.5 | Suspiciously Predictive |

## Why Are WoE and IV Important?

- Widely used in credit scoring systems
- Improve model interpretability
- Support regulatory requirements such as Basel II
- Help identify important predictive variables

## Implementation Note

For demonstration purposes, a WoE implementation was explored using FraudResult as a temporary target variable. However, the dataset does not contain a true default label. Therefore, the final WoE transformation will be performed after creating an RFM-based proxy default target variable in later stages of the project.